<a href="https://colab.research.google.com/github/SenTier1107/Appprogramming_2026/blob/main/0417_FastAPI_Prerequisites/04_17_%EC%8B%A4%EC%8A%B5%EA%B3%BC%EC%A0%9C(1%2C2%2C3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📝 실습과제 모음

---

| # | 출처 | 과제 내용 |
|---|------|----------|
| 실습과제 1 | `04_sqlite3_introduction` | CSV → SQLite3 CRUD |
| 실습과제 2 | `04_sqlite3_introduction` | 모듈화 리팩토링 (`init`, `create`, `read`, `update`, `delete`) |
| 실습과제 3 | `05_sqlite3_pydantic` | CSV → Pydantic 모델로 저장 |

---
# 📌 실습과제 1 — CSV → SQLite3 CRUD

**과제 내용**
> `https://github.com/ancestor9/data/blob/main/customers.csv` 를 읽어와서
> 데이터베이스를 만들고 `customers` 테이블에 CRUD하는 코드를 작성하라.

In [ ]:
!pip install pandas --quiet
print('설치 완료')

설치 완료


In [ ]:
import pandas as pd
import sqlite3
import os

CUSTOMER_DB_NAME = "customer_data.db"

def init_customer_db():
    """고객 데이터베이스 초기화 — 기존 파일 삭제 후 테이블 새로 생성"""
    if os.path.exists(CUSTOMER_DB_NAME):
        os.remove(CUSTOMER_DB_NAME)
        print(f"이전 DB 파일 삭제: {CUSTOMER_DB_NAME}")

    conn = sqlite3.connect(CUSTOMER_DB_NAME)
    cursor = conn.cursor()

    print("\n── [CREATE] customers 테이블 생성 ──")
    cursor.execute("""
        CREATE TABLE customers (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE,
            full_name TEXT
        )
    """)
    conn.commit()
    print("✅ 테이블 생성 완료")
    return conn

conn = init_customer_db()
cursor = conn.cursor()


── [CREATE] customers 테이블 생성 ──
✅ 테이블 생성 완료


In [ ]:
# ── [C] Create: CSV 상위 10개 데이터 삽입 ──
print("── [C] Create ──")

csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"
df = pd.read_csv(csv_url)
df_top10 = df.head(10)  # 상위 10개만 사용

customer_data = [
    (row['고객ID'], row['고객ID'])
    for _, row in df_top10.iterrows()
]

cursor.executemany(
    "INSERT INTO customers (username, full_name) VALUES (?, ?)",
    customer_data
)
conn.commit()
print(f"✅ {len(customer_data)}개 데이터 삽입 완료")

── [C] Create ──
✅ 10개 데이터 삽입 완료


In [ ]:
# ── [R] Read: 전체 조회 ──
print("── [R] Read: 전체 조회 ──")

cursor.execute("SELECT id, username, full_name FROM customers LIMIT 10")
for row in cursor.fetchall():
    print(f"  ID: {row[0]}, Username: {row[1]}, FullName: {row[2]}")

── [R] Read: 전체 조회 ──
  ID: 1, Username: CUST_0001, FullName: CUST_0001
  ID: 2, Username: CUST_0002, FullName: CUST_0002
  ID: 3, Username: CUST_0003, FullName: CUST_0003
  ID: 4, Username: CUST_0004, FullName: CUST_0004
  ID: 5, Username: CUST_0005, FullName: CUST_0005
  ID: 6, Username: CUST_0006, FullName: CUST_0006
  ID: 7, Username: CUST_0007, FullName: CUST_0007
  ID: 8, Username: CUST_0008, FullName: CUST_0008
  ID: 9, Username: CUST_0009, FullName: CUST_0009
  ID: 10, Username: CUST_0010, FullName: CUST_0010


In [ ]:
# ── [U] Update: CUST_0002 수정 ──
print("── [U] Update: CUST_0002 수정 ──")

cursor.execute(
    "UPDATE customers SET full_name = ? WHERE username = ?",
    ('Updated User 02', 'CUST_0002')
)
conn.commit()

cursor.execute("SELECT * FROM customers WHERE username = 'CUST_0002'")
print(f"  수정 결과: {cursor.fetchone()}")

── [U] Update: CUST_0002 수정 ──
  수정 결과: (2, 'CUST_0002', 'Updated User 02')


In [ ]:
# ── [D] Delete: CUST_0003 삭제 ──
print("── [D] Delete: CUST_0003 삭제 ──")

cursor.execute(
    "DELETE FROM customers WHERE username = ?",
    ('CUST_0003',)
)
conn.commit()

cursor.execute("SELECT * FROM customers WHERE username = 'CUST_0003'")
result = cursor.fetchone()
print(f"  삭제 확인: {'삭제됨 (None)' if result is None else result}")

# ── 최종 결과 확인 ──
print("\n── 최종 데이터 ──")
cursor.execute("SELECT * FROM customers ORDER BY id ASC")
for row in cursor.fetchall():
    print(f"  {row}")

conn.close()
print("\n✅ 실습과제 1 완료")

── [D] Delete: CUST_0003 삭제 ──
  삭제 확인: 삭제됨 (None)

── 최종 데이터 ──
  (1, 'CUST_0001', 'CUST_0001')
  (2, 'CUST_0002', 'Updated User 02')
  (4, 'CUST_0004', 'CUST_0004')
  (5, 'CUST_0005', 'CUST_0005')
  (6, 'CUST_0006', 'CUST_0006')
  (7, 'CUST_0007', 'CUST_0007')
  (8, 'CUST_0008', 'CUST_0008')
  (9, 'CUST_0009', 'CUST_0009')
  (10, 'CUST_0010', 'CUST_0010')

✅ 실습과제 1 완료




---



---



---



In [ ]:
"""
추가 실습과제 1 — Gradio CRUD Dashboard
customers.csv 기반 | 고객ID 자동 배정 | Pydantic 유효성 검사 | Gradio Blocks
"""

# ── 설치 (Colab 환경) ──
# !pip install gradio pydantic pandas --quiet

import sqlite3
import os
import re
import pandas as pd
import gradio as gr
from pydantic import BaseModel, field_validator, ValidationError

# 기존 세션 종료 + DB 초기화
gr.close_all()
if os.path.exists("customer_gradio.db"):
    os.remove("customer_gradio.db")

# ════════════════════════════════════════
# 1. 설정
# ════════════════════════════════════════
DB_NAME = "customer_gradio.db"
CSV_URL = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"


# ════════════════════════════════════════
# 2. Pydantic 모델
# ════════════════════════════════════════
class CustomerInput(BaseModel):
    """Create/Update 입력값 검증"""
    성별: str
    결제수단: str
    거주지: str
    회원등급: str
    만족도: int
    선호온도: float
    나이: int

    @field_validator('성별', '결제수단', '거주지', '회원등급')
    @classmethod
    def no_digit_in_text(cls, v, info):
        v = v.strip() if isinstance(v, str) else str(v)
        if not v:
            raise ValueError(f"'{info.field_name}' 항목은 비워둘 수 없습니다.")
        if any(c.isdigit() for c in v):
            raise ValueError(f"'{info.field_name}' 항목에 숫자를 포함할 수 없습니다.")
        return v

    @field_validator('나이', mode='before')
    @classmethod
    def validate_age(cls, v):
        if isinstance(v, str):
            v = v.strip()
            if not v:
                raise ValueError("나이를 입력해주세요.")
            if not v.lstrip('-').isdigit():
                raise ValueError(f"나이는 숫자만 입력 가능합니다. (입력값: '{v}')")
            v = int(v)
        try:
            v = int(v)
        except (ValueError, TypeError):
            raise ValueError("나이는 숫자만 입력 가능합니다.")
        if v <= 0 or v > 120:
            raise ValueError("나이는 1~120 사이의 값만 입력 가능합니다.")
        return v

    @field_validator('만족도', mode='before')
    @classmethod
    def validate_satisfaction(cls, v):
        if isinstance(v, str):
            v = v.strip()
            if not v:
                raise ValueError("만족도를 입력해주세요.")
            if not v.lstrip('-').isdigit():
                raise ValueError(f"만족도는 숫자만 입력 가능합니다. (입력값: '{v}')")
            v = int(v)
        try:
            v = int(v)
        except (ValueError, TypeError):
            raise ValueError("만족도는 숫자만 입력 가능합니다.")
        if not (1 <= v <= 5):
            raise ValueError("만족도는 1~5 사이의 값만 입력 가능합니다.")
        return v

    @field_validator('선호온도', mode='before')
    @classmethod
    def validate_temp(cls, v):
        if isinstance(v, str):
            v = v.strip()
            if not v:
                raise ValueError("선호온도를 입력해주세요.")
            try:
                v = float(v)
            except ValueError:
                raise ValueError(f"선호온도는 숫자만 입력 가능합니다. (입력값: '{v}')")
        try:
            v = float(v)
        except (ValueError, TypeError):
            raise ValueError("선호온도는 숫자만 입력 가능합니다.")
        if v == 0:
            raise ValueError("선호온도를 입력해주세요. (-10~50 범위)")
        if v < -10 or v > 50:
            raise ValueError("선호온도는 -10~50 범위의 값만 입력 가능합니다.")
        return v


# ════════════════════════════════════════
# 3. DB 초기화
# ════════════════════════════════════════
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id           INTEGER PRIMARY KEY AUTOINCREMENT,
            고객ID       TEXT NOT NULL UNIQUE,
            성별         TEXT,
            결제수단     TEXT,
            거주지       TEXT,
            회원등급     TEXT,
            만족도       INTEGER,
            최근접속시간  INTEGER,
            선호온도     REAL,
            나이         INTEGER,
            구매수량     INTEGER,
            총결제금액   INTEGER
        )
    """)
    cursor.execute("SELECT COUNT(*) FROM customers")
    if cursor.fetchone()[0] == 0:
        df = pd.read_csv(CSV_URL).head(10)
        df = df.rename(columns={
            '최근접속시간(시)': '최근접속시간',
            '선호제품군_적정온도': '선호온도'
        })
        data = [
            (row['고객ID'], row['성별'], row['결제수단'], row['거주지'],
             row['회원등급'], int(row['만족도']), int(row['최근접속시간']),
             float(row['선호온도']), int(row['나이']),
             int(row['구매수량']), int(row['총결제금액']))
            for _, row in df.iterrows()
        ]
        cursor.executemany("""
            INSERT INTO customers
            (고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 최근접속시간, 선호온도, 나이, 구매수량, 총결제금액)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """, data)
        print(f"✅ CSV 초기 데이터 {len(data)}개 로드 완료")
    conn.commit()
    conn.close()

init_db()


# ════════════════════════════════════════
# 4. 헬퍼 함수
# ════════════════════════════════════════
def get_conn():
    return sqlite3.connect(DB_NAME)

DISP_COLS = "id, 고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이"

def read_df() -> pd.DataFrame:
    """표시용 컬럼만 조회 (구매수량·접속시간·결제금액 제외)"""
    conn = get_conn()
    df = pd.read_sql_query(
        f"SELECT {DISP_COLS} FROM customers ORDER BY id ASC", conn
    )
    conn.close()
    return df

def next_customer_id() -> str:
    """현재 최대 번호 + 1로 다음 고객ID 자동 생성 (예: CUST_0012)"""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT 고객ID FROM customers ORDER BY id DESC LIMIT 1")
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return "CUST_0001"
    match = re.search(r'\d+', row[0])
    next_num = int(match.group()) + 1 if match else 1
    return f"CUST_{next_num:04d}"

def get_summary() -> str:
    """대시보드 상단 요약 통계"""
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*), AVG(만족도), AVG(나이) FROM customers")
    row = cursor.fetchone()
    conn.close()
    total, avg_sat, avg_age = row
    return (f"👥 총 고객: **{total}명**  |  "
            f"⭐ 평균 만족도: **{avg_sat:.1f}**  |  "
            f"🎂 평균 나이: **{avg_age:.1f}세**")


# ════════════════════════════════════════
# 5. CRUD 함수
# ════════════════════════════════════════
def db_create(성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이):
    try:
        d = CustomerInput(성별=성별, 결제수단=결제수단, 거주지=거주지,
                          회원등급=회원등급, 만족도=만족도, 선호온도=선호온도, 나이=나이)
    except ValidationError as e:
        msg = e.errors()[0]['msg'].replace('Value error, ', '')
        return f"❌ {msg}", read_df(), get_summary(), next_customer_id()

    new_id = next_customer_id()
    conn = get_conn()
    try:
        conn.execute("""
            INSERT INTO customers (고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (new_id, d.성별, d.결제수단, d.거주지, d.회원등급, d.만족도, d.선호온도, d.나이))
        conn.commit()
        return f"✅ 추가 완료: {new_id}", read_df(), get_summary(), next_customer_id()
    except sqlite3.IntegrityError:
        return f"❌ 중복 오류가 발생했습니다.", read_df(), get_summary(), next_customer_id()
    finally:
        conn.close()


def db_read(search: str):
    conn = get_conn()
    if search.strip():
        df = pd.read_sql_query(
            f"SELECT {DISP_COLS} FROM customers "
            f"WHERE 고객ID LIKE ? OR 거주지 LIKE ? OR 회원등급 LIKE ? OR 성별 LIKE ? "
            f"ORDER BY id ASC",
            conn, params=(f"%{search}%",) * 4
        )
        msg = f"🔍 '{search}' 검색 결과: {len(df)}건"
    else:
        df = pd.read_sql_query(
            f"SELECT {DISP_COLS} FROM customers ORDER BY id ASC", conn
        )
        msg = f"📋 전체 조회: {len(df)}건"
    conn.close()
    return msg, df


def load_customer(고객ID: str):
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요.", "", "", "", "", 0, 0.0, 0
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이 "
        "FROM customers WHERE 고객ID=?", (고객ID,)
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", "", "", "", "", 0, 0.0, 0
    return (f"✅ '{고객ID}' 불러오기 완료", row[0], row[1], row[2], row[3], row[4], row[5], row[6])


def db_update(고객ID, 성별, 결제수단, 거주지, 회원등급, 만족도, 선호온도, 나이):
    try:
        d = CustomerInput(성별=성별, 결제수단=결제수단, 거주지=거주지,
                          회원등급=회원등급, 만족도=만족도, 선호온도=선호온도, 나이=나이)
    except ValidationError as e:
        msg = e.errors()[0]['msg'].replace('Value error, ', '')
        return f"❌ {msg}", read_df(), get_summary()

    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("""
        UPDATE customers SET 성별=?, 결제수단=?, 거주지=?, 회원등급=?, 만족도=?, 선호온도=?, 나이=?
        WHERE 고객ID=?
    """, (d.성별, d.결제수단, d.거주지, d.회원등급, d.만족도, d.선호온도, d.나이, 고객ID))
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    if affected == 0:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", read_df(), get_summary()
    return f"✅ 수정 완료: {고객ID}", read_df(), get_summary()


def db_delete(고객ID: str):
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요.", read_df(), get_summary()
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute("DELETE FROM customers WHERE 고객ID=?", (고객ID,))
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    if affected == 0:
        return f"❌ '{고객ID}'를 찾을 수 없습니다.", read_df(), get_summary()
    return f"✅ 삭제 완료: {고객ID}", read_df(), get_summary()


def greet_user(고객ID: str) -> str:
    고객ID = 고객ID.strip()
    if not 고객ID:
        return "❌ 고객ID를 입력해주세요."
    conn = get_conn()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT 성별, 거주지, 회원등급, 나이, 만족도 FROM customers WHERE 고객ID=?", (고객ID,)
    )
    row = cursor.fetchone()
    conn.close()
    if row is None:
        return f"❌ '{고객ID}'를 찾을 수 없습니다."
    return (f"👋 Hello, {고객ID}!\n\n"
            f"  성별     : {row[0]}\n"
            f"  거주지   : {row[1]}\n"
            f"  회원등급 : {row[2]}\n"
            f"  나이     : {row[3]}세\n"
            f"  만족도   : {'⭐' * row[4]} ({row[4]}/5)")


# ════════════════════════════════════════
# 6. 선택 옵션 상수
# ════════════════════════════════════════
OPT_GENDER  = ["남성", "여성"]
OPT_GRADE   = ["Bronze", "Silver", "Gold", "VIP", "VVIP"]
OPT_PAYMENT = ["신용카드", "계좌이체", "휴대폰결제", "간편결제"]
OPT_REGION  = ["서울", "부산", "대구", "광주", "인천", "경기", "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주"]


# ════════════════════════════════════════
# 7. Gradio UI
# ════════════════════════════════════════
css = """
.tab-nav button { font-size: 15px !important; font-weight: 600 !important; }
.status-box textarea { font-size: 14px !important; }
.summary-box { font-size: 15px !important; }

/* 모든 textarea 화살표·크기조절 완전 제거 */
textarea { resize: none !important; overflow: hidden !important; }
div[data-testid="textbox"] textarea { resize: none !important; overflow: hidden !important; }
"""

with gr.Blocks(title="Customer CRUD Dashboard", theme=gr.themes.Soft(), css=css) as demo:

    # ── 헤더 ──
    gr.Markdown("# 🗄️ Customer CRUD Dashboard")
    gr.Markdown("---")
    summary_md = gr.Markdown(value=get_summary(), elem_classes="summary-box")
    gr.Markdown("---")

    with gr.Tabs():

        # ════════ ➕ Create ════════
        with gr.Tab("➕  Create"):
            gr.Markdown("### 새 고객 등록")
            gr.Markdown("> 고객ID는 **자동 배정**됩니다. 드롭다운에서 선택하거나 직접 입력하세요.")

            with gr.Row():
                auto_id = gr.Textbox(
                    label="자동 배정될 고객ID",
                    value=next_customer_id,
                    interactive=False,
                    lines=1,
                    max_lines=1,
                    scale=1
                )

            gr.Markdown("**기본 정보**")
            with gr.Row():
                c_gender  = gr.Dropdown(label="성별",     choices=OPT_GENDER,  allow_custom_value=True, scale=1)
                c_grade   = gr.Dropdown(label="회원등급", choices=OPT_GRADE,   allow_custom_value=True, scale=1)
                c_payment = gr.Dropdown(label="결제수단", choices=OPT_PAYMENT, allow_custom_value=True, scale=2)
                c_region  = gr.Dropdown(label="거주지",   choices=OPT_REGION,  allow_custom_value=True, scale=2)

            gr.Markdown("**수치 정보**")
            with gr.Row():
                c_age  = gr.Number(label="나이",          value=0, minimum=0,  maximum=120, step=1,   precision=0, scale=1)
                c_sat  = gr.Number(label="만족도 (1~5)",  value=0, step=1, precision=0, scale=1)
                c_temp = gr.Number(label="선호온도 (℃)", value=0, minimum=-10, maximum=50,  step=0.5,              scale=1)

            with gr.Row():
                c_btn = gr.Button("➕  고객 추가", variant="primary", scale=1)

            c_status = gr.Textbox(label="결과", interactive=False, lines=1, max_lines=1, elem_classes="status-box")
            c_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            c_btn.click(
                fn=db_create,
                inputs=[c_gender, c_payment, c_region, c_grade, c_sat, c_temp, c_age],
                outputs=[c_status, c_table, summary_md, auto_id]
            )

        # ════════ 🔍 Read ════════
        with gr.Tab("🔍  Read"):
            gr.Markdown("### 고객 조회")
            gr.Markdown("💡 **검색 가능 항목**: 고객ID · 거주지 · 회원등급 · 성별  |  비워두면 전체 조회됩니다.")
            with gr.Row():
                r_search = gr.Textbox(
                    label="검색",
                    lines=1,
                    max_lines=1,
                    placeholder="예) CUST_0001  /  서울  /  Gold  /  남성",
                    scale=4
                )
                r_btn = gr.Button("🔍  조회", variant="primary", scale=1)
            r_status = gr.Textbox(label="결과", interactive=False, lines=1, max_lines=1, elem_classes="status-box")
            r_table  = gr.DataFrame(label="조회 결과", value=read_df)

            r_btn.click(fn=db_read, inputs=[r_search], outputs=[r_status, r_table])

        # ════════ ✏️ Update ════════
        with gr.Tab("✏️  Update"):
            gr.Markdown("### 고객 정보 수정")
            gr.Markdown("> **① 고객ID 입력 → 불러오기** &nbsp;→&nbsp; **② 값 수정** &nbsp;→&nbsp; **③ 수정하기**")

            with gr.Row():
                u_id       = gr.Textbox(label="고객ID", placeholder="예) CUST_0001", lines=1, max_lines=1, scale=4)
                u_load_btn = gr.Button("📂  불러오기", variant="secondary", scale=1)
            u_load_status = gr.Textbox(label="불러오기 결과", interactive=False, lines=1, max_lines=1, elem_classes="status-box")

            gr.Markdown("**기본 정보**")
            with gr.Row():
                u_gender  = gr.Dropdown(label="성별",     choices=OPT_GENDER,  allow_custom_value=True, scale=1)
                u_grade   = gr.Dropdown(label="회원등급", choices=OPT_GRADE,   allow_custom_value=True, scale=1)
                u_payment = gr.Dropdown(label="결제수단", choices=OPT_PAYMENT, allow_custom_value=True, scale=2)
                u_region  = gr.Dropdown(label="거주지",   choices=OPT_REGION,  allow_custom_value=True, scale=2)

            gr.Markdown("**수치 정보**")
            with gr.Row():
                u_age  = gr.Number(label="나이",          value=0, minimum=0,   maximum=120, step=1,   precision=0, scale=1)
                u_sat  = gr.Number(label="만족도 (1~5)",  value=0, step=1, precision=0, scale=1)
                u_temp = gr.Number(label="선호온도 (℃)", value=0, minimum=-10,  maximum=50,  step=0.5,              scale=1)

            with gr.Row():
                u_btn = gr.Button("✏️  수정하기", variant="primary", scale=1)

            u_status = gr.Textbox(label="결과", interactive=False, lines=1, max_lines=1, elem_classes="status-box")
            u_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            u_load_btn.click(
                fn=load_customer,
                inputs=[u_id],
                outputs=[u_load_status, u_gender, u_payment, u_region, u_grade, u_sat, u_temp, u_age]
            )
            u_btn.click(
                fn=db_update,
                inputs=[u_id, u_gender, u_payment, u_region, u_grade, u_sat, u_temp, u_age],
                outputs=[u_status, u_table, summary_md]
            )

        # ════════ 👋 Greet ════════
        with gr.Tab("👋  Greet"):
            gr.Markdown("### 고객 인사말")
            gr.Markdown("> 고객ID를 입력하면 해당 고객 정보와 함께 인사말을 출력합니다.")
            with gr.Row():
                g_id  = gr.Textbox(label="고객ID", placeholder="예) CUST_0001", lines=1, max_lines=1, scale=4)
                g_btn = gr.Button("👋  인사하기", variant="primary", scale=1)
            g_output = gr.Textbox(label="결과", lines=7, interactive=False, elem_classes="status-box")

            g_btn.click(fn=greet_user, inputs=[g_id], outputs=[g_output])

        # ════════ 🗑️ Delete ════════
        with gr.Tab("🗑️  Delete"):
            gr.Markdown("### 고객 삭제")
            gr.Markdown("> ⚠️ 삭제된 데이터는 복구되지 않습니다.")
            with gr.Row():
                d_id  = gr.Textbox(label="삭제할 고객ID", placeholder="예) CUST_0003", lines=1, max_lines=1, scale=4)
                d_btn = gr.Button("🗑️  삭제하기", variant="stop", scale=1)
            d_status = gr.Textbox(label="결과", interactive=False, lines=1, max_lines=1, elem_classes="status-box")
            d_table  = gr.DataFrame(label="전체 고객 목록", value=read_df)

            d_btn.click(fn=db_delete, inputs=[d_id], outputs=[d_status, d_table, summary_md])

demo.launch()



---



---



---



---
# 📌 실습과제 2 — 모듈화 리팩토링

**과제 내용**
> 실습과제 1의 코드를 `init.py`, `create.py`, `read.py`, `update.py`, `delete.py`로
> 모듈화하여 실행하는 코드로 리팩토링하라.

각 모듈의 역할:

| 파일 | 역할 |
|------|------|
| `init.py` | DB 연결 및 테이블 생성 |
| `create.py` | 데이터 삽입 함수 |
| `read.py` | 데이터 조회 함수 |
| `update.py` | 데이터 수정 함수 |
| `delete.py` | 데이터 삭제 함수 |

In [ ]:
# ── init.py ──
# DB 연결 및 테이블 생성 모듈

import sqlite3
import os

DB_NAME = "customer_modular.db"

def get_connection():
    """DB 연결 객체 반환"""
    return sqlite3.connect(DB_NAME)

def init_db():
    """DB 초기화 — 테이블이 없으면 생성"""
    if os.path.exists(DB_NAME):
        os.remove(DB_NAME)

    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS customers (
            id       INTEGER PRIMARY KEY AUTOINCREMENT,
            username TEXT NOT NULL UNIQUE,
            full_name TEXT
        )
    """)
    conn.commit()
    conn.close()
    print("[init] ✅ DB 초기화 완료")

init_db()

[init] ✅ DB 초기화 완료


In [ ]:
# ── create.py ──
# 데이터 삽입 모듈

import pandas as pd

def create_customers(data: list[tuple]):
    """customers 테이블에 여러 행 삽입"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT INTO customers (username, full_name) VALUES (?, ?)",
        data
    )
    conn.commit()
    conn.close()
    print(f"[create] ✅ {len(data)}개 삽입 완료")

# CSV에서 상위 10개 로드 후 삽입
csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"
df = pd.read_csv(csv_url).head(10)
data = [(row['고객ID'], row['고객ID']) for _, row in df.iterrows()]
create_customers(data)

[create] ✅ 10개 삽입 완료


In [ ]:
# ── read.py ──
# 데이터 조회 모듈

def read_all_customers():
    """전체 고객 조회"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT id, username, full_name FROM customers")
    rows = cursor.fetchall()
    conn.close()
    return rows

def read_customer_by_username(username: str):
    """username으로 특정 고객 조회"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT * FROM customers WHERE username = ?",
        (username,)
    )
    row = cursor.fetchone()
    conn.close()
    return row

# 실행
print("[read] 전체 고객 조회")
for row in read_all_customers():
    print(f"  {row}")

[read] 전체 고객 조회
  (1, 'CUST_0001', 'CUST_0001')
  (2, 'CUST_0002', 'CUST_0002')
  (3, 'CUST_0003', 'CUST_0003')
  (4, 'CUST_0004', 'CUST_0004')
  (5, 'CUST_0005', 'CUST_0005')
  (6, 'CUST_0006', 'CUST_0006')
  (7, 'CUST_0007', 'CUST_0007')
  (8, 'CUST_0008', 'CUST_0008')
  (9, 'CUST_0009', 'CUST_0009')
  (10, 'CUST_0010', 'CUST_0010')


In [ ]:
# ── update.py ──
# 데이터 수정 모듈

def update_customer_fullname(username: str, new_fullname: str):
    """특정 고객의 full_name 수정"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE customers SET full_name = ? WHERE username = ?",
        (new_fullname, username)
    )
    conn.commit()
    conn.close()
    print(f"[update] ✅ {username} → full_name = '{new_fullname}'")

# 실행
update_customer_fullname('CUST_0002', 'Updated User 02')
print(f"  수정 확인: {read_customer_by_username('CUST_0002')}")

[update] ✅ CUST_0002 → full_name = 'Updated User 02'
  수정 확인: (2, 'CUST_0002', 'Updated User 02')


In [ ]:
# ── delete.py ──
# 데이터 삭제 모듈

def delete_customer(username: str):
    """특정 고객 삭제"""
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "DELETE FROM customers WHERE username = ?",
        (username,)
    )
    conn.commit()
    conn.close()
    print(f"[delete] ✅ {username} 삭제 완료")

# 실행
delete_customer('CUST_0003')
result = read_customer_by_username('CUST_0003')
print(f"  삭제 확인: {'없음 (삭제됨)' if result is None else result}")

# ── 최종 결과 ──
print("\n── 최종 데이터 ──")
for row in read_all_customers():
    print(f"  {row}")

print("\n✅ 실습과제 2 완료")

[delete] ✅ CUST_0003 삭제 완료
  삭제 확인: 없음 (삭제됨)

── 최종 데이터 ──
  (1, 'CUST_0001', 'CUST_0001')
  (2, 'CUST_0002', 'Updated User 02')
  (4, 'CUST_0004', 'CUST_0004')
  (5, 'CUST_0005', 'CUST_0005')
  (6, 'CUST_0006', 'CUST_0006')
  (7, 'CUST_0007', 'CUST_0007')
  (8, 'CUST_0008', 'CUST_0008')
  (9, 'CUST_0009', 'CUST_0009')
  (10, 'CUST_0010', 'CUST_0010')

✅ 실습과제 2 완료


---
# 📌 실습과제 3 — CSV → Pydantic 모델로 저장

**과제 내용**
> `https://github.com/ancestor9/data/blob/main/customers.csv` 를 읽어와서
> Pydantic 모델에 저장하라.

- `Field(alias=...)` 로 한글 컬럼명을 영문 속성명으로 매핑
- `model_validate()` 로 DataFrame → Pydantic 모델 변환
- `model_dump_json()` 으로 JSON 직렬화 확인

In [ ]:
!pip install pydantic pandas --quiet
print('설치 완료')

설치 완료


In [ ]:
import pandas as pd
from pydantic import BaseModel, Field
from typing import List

# ── Pydantic 모델 정의 ──
# Field(alias='한글컬럼명') 으로 CSV 컬럼명과 매핑
class Customer(BaseModel):
    customer_id: str           = Field(alias='고객ID')
    gender: str                = Field(alias='성별')
    payment_method: str        = Field(alias='결제수단')
    residence: str             = Field(alias='거주지')
    membership_level: str      = Field(alias='회원등급')
    satisfaction: int          = Field(alias='만족도')
    recent_access_hour: int    = Field(alias='최근접속시간(시)')
    preferred_temp: float      = Field(alias='선호제품군_적정온도')
    age: int                   = Field(alias='나이')
    purchase_quantity: int     = Field(alias='구매수량')
    total_purchase_amount: int = Field(alias='총결제금액')

print("✅ Customer 모델 정의 완료")

✅ Customer 모델 정의 완료


In [ ]:
# ── CSV 로드 ──
csv_url = "https://raw.githubusercontent.com/ancestor9/data/main/customers.csv"
df = pd.read_csv(csv_url)

print(f"✅ CSV 로드 완료: {len(df)}개 레코드")
print(f"   컬럼: {df.columns.tolist()}")

✅ CSV 로드 완료: 10000개 레코드
   컬럼: ['고객ID', '성별', '결제수단', '거주지', '회원등급', '만족도', '최근접속시간(시)', '선호제품군_적정온도', '나이', '구매수량', '총결제금액']


In [ ]:
# ── DataFrame → Pydantic 모델 변환 ──
# to_dict('records'): DataFrame을 dict 리스트로 변환
# model_validate(): dict → Pydantic 모델 인스턴스로 검증 및 변환
customers: List[Customer] = [
    Customer.model_validate(record)
    for record in df.to_dict('records')
]

print(f"✅ Pydantic 변환 완료: {len(customers)}개")

✅ Pydantic 변환 완료: 10000개


In [ ]:
# ── 결과 확인 ──
print("── 상위 3개 고객 (JSON 직렬화) ──")
for c in customers[:3]:
    print(c.model_dump_json(indent=2))
    print()

# 도트 연산자로 접근 확인
print("── 도트 연산자 접근 ──")
for c in customers[:5]:
    print(f"  고객ID: {c.customer_id} | 나이: {c.age} | 등급: {c.membership_level} | 만족도: {c.satisfaction}")

print("\n✅ 실습과제 3 완료")

── 상위 3개 고객 (JSON 직렬화) ──
{
  "customer_id": "CUST_0001",
  "gender": "남성",
  "payment_method": "휴대폰결제",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 2,
  "recent_access_hour": 1,
  "preferred_temp": 27.0,
  "age": 34,
  "purchase_quantity": 7,
  "total_purchase_amount": 760355
}

{
  "customer_id": "CUST_0002",
  "gender": "여성",
  "payment_method": "계좌이체",
  "residence": "대구",
  "membership_level": "Gold",
  "satisfaction": 5,
  "recent_access_hour": 15,
  "preferred_temp": 24.6,
  "age": 20,
  "purchase_quantity": 42,
  "total_purchase_amount": 727001
}

{
  "customer_id": "CUST_0003",
  "gender": "여성",
  "payment_method": "신용카드",
  "residence": "광주",
  "membership_level": "Gold",
  "satisfaction": 1,
  "recent_access_hour": 6,
  "preferred_temp": 20.3,
  "age": 51,
  "purchase_quantity": 8,
  "total_purchase_amount": 618787
}

── 도트 연산자 접근 ──
  고객ID: CUST_0001 | 나이: 34 | 등급: Gold | 만족도: 2
  고객ID: CUST_0002 | 나이: 20 | 등급: Gold | 만족도: 5
  고객ID: CUST_0003 | 나이: 